In [3]:
#Final Project
#Hello guys

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# Load the Data

data = pd.read_csv("Song Recommendations.csv")

#data.describe()

FileNotFoundError: [Errno 2] No such file or directory: 'Song Recommendations.csv'

In [ ]:
# Data Pre-processing - fixing category name descrepancies

#data.describe()
#data['Genre'].value_counts()
data['Language of the song (select all that applied)'].value_counts()

data['Genre'] = data['Genre'].replace('Videogame Music', 'Video Game')
data['Genre'] = data['Genre'].replace('Rock + Opera + Ballad', 'Rock / Opera / Ballad')
data['Language of the song (select all that applied)'] = data['Language of the song (select all that applied)'].replace('No lyrics', 'Instrumental')
data['Language of the song (select all that applied)'] = data['Language of the song (select all that applied)'].replace('No Language', 'Instrumental')
data['Language of the song (select all that applied)'] = data['Language of the song (select all that applied)'].replace('No language', 'Instrumental')

In [ ]:
# Data Pre-processing - dealing with missing values

# data.isna().describe()
# only a few missing so just dropped rows with missing values
data = data.dropna()

In [ ]:
# Data Pre-processing - Label Encoding for Song release year

# get all column names for song year

data['Song release year'].value_counts()
SongYearDic = {'Pre-1960': 0, '1960–1979': 1, '1980–1999': 2, '2000–2009': 3, '2010–2019': 4, '2020+': 5}
data['Song_year_encoded'] = data['Song release year'].map(SongYearDic)

In [ ]:
# Data Pre-Processing - One hot Encoding for Gender and Hometown

data = pd.get_dummies(data, columns = ['Your Gender'], dtype = int)
data = pd.get_dummies(data, columns = ['Your hometown'], dtype = int)

In [ ]:
# Data Pre-Processing - Multi hot Encoding for Genre and Languages

# create lists for genre and song language
data['Genre'] = data['Genre'].str.split('/')
data['Language of the song (select all that applied)'] = data['Language of the song (select all that applied)'].str.split(';')

# multi-hot encoding from scikit learn

from sklearn.preprocessing import MultiLabelBinarizer
mlb_genre = MultiLabelBinarizer()
arr_genre = mlb_genre.fit_transform(data['Genre'])
genre_df = pd.DataFrame(arr_genre, index = data.index, columns = mlb_genre.classes_)

mlb_langs = MultiLabelBinarizer()
arr_langs = mlb_langs.fit_transform(data['Language of the song (select all that applied)'])
langs_df = pd.DataFrame(arr_langs, index = data.index, columns = mlb_langs.classes_)
langs_df = langs_df.add_suffix('_song')

mlb_langh = MultiLabelBinarizer()
arr_langh = mlb_langh.fit_transform(data['Language of the song (select all that applied)'])
langh_df = pd.DataFrame(arr_langh, index = data.index, columns = mlb_langh.classes_)
langh_df = langh_df.add_suffix('_home')

# combine into 1 df

data_enc = pd.concat([data, genre_df], axis = 1)
data_enc = pd.concat([data_enc, langs_df], axis = 1)
data_enc = pd.concat([data_enc, langh_df], axis = 1)

In [ ]:
# drop extra/unencoded columns

data_enc_dropped = data_enc.drop(columns = ['Genre']).drop(columns = ['Language of the song (select all that applied)']).drop(columns = ['What language do you primarily use in daily life? select all that applied']).drop(columns = ['Song release year'])

# data_enc_dropped has all categorical variables encoded and dropped the original columns so there wouldn't be duplicates
# data_enc has all categorical variables but still includes original columns

# data_enc_dropped.head()

In [ ]:
#importing necessary packages for KNN
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
#Using KNN to test different K and see which is most accurate

X = data_enc_dropped.select_dtypes(include=[np.number])
y = genre_df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

K_list = [1, 3, 5, 10, 15, 20, 25, 30]
accs = []

for K in K_list:
    knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=K))
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = np.mean((y_pred == y_test.values).all(axis=1))
    print(f"K = {K}, Accuracy = {acc:.4f}")
    accs.append(acc)